# Phase 6: Exploratory Data Analysis (EDA) & Data Visualization

This notebook focuses on extracting actionable business insights from the final Master Dataset (`sales_full_dataset.csv`). Using interactive visualizations, we will analyze sales performance across multiple dimensions to support data-driven decision-making.

**Key Analytical Objectives:**
* **Geographic & Product Analysis:** Identifying top-revenue cities, average deal sizes, and the highest-contributing projects.
* **Trend & Channel Evaluation:** Tracking monthly sales growth and comparing revenue across different sales channels.
* **Agent & Customer Insights:** Assessing individual sales agent performance and understanding revenue distribution by customer segment.

In [2]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("sales_full_dataset.csv")
df.head()

,sale_id,sale_date,customer_id,project_id,unit_type,unit_price,discount,final_price,payment_plan,down_payment,...,developer,project_type,start_price,max_price,units_count,delivery_year,status_y,price_range,avg_price,price_category
0,S2023010000000,2023-01-14,CUST_22585,PRJ_30,3Br,6691492,100000,6591492,8Y,938086.13,...,City Edge,Residential,1936251,5221584,562,2026,Under Development,3285333,3578917.5,Medium
1,S2023010000001,2023-01-17,CUST_23621,PRJ_19,Studio,2846816,100000,2746816,Cash,313545.17,...,Palm Hills,Residential,2580461,8735641,418,2025,Delivered,6155180,5658051.0,High
2,S2023010000002,2023-01-19,CUST_12281,PRJ_4,Villa,1357382,0,1357382,8Y,196379.95,...,Sodic,Residential,2517796,5379090,289,2028,Delivered,2861294,3948443.0,Medium
3,S2023010000003,2023-01-11,CUST_8266,PRJ_18,2Br,2363226,100000,2263226,Cash,554752.04,...,City Edge,Administrative,2509223,9696290,870,2027,Under Construction,7187067,6102756.5,High
4,S2023010000004,2023-01-18,CUST_14652,PRJ_34,3Br,5212700,0,5212700,5Y,1536354.55,...,Palm Hills,Residential,2080403,5674636,421,2028,Under Construction,3594233,3877519.5,Medium


In [3]:
# Calculate total revenue (final_price) per city and sort descending to identify top-performing regions
city_sales = (
    df.groupby('city')['final_price']
      .sum()
      .reset_index()
      .sort_values('final_price', ascending=False)
)
city_sales

,city,final_price
0,Arbeen,888560904348
1,Assiut East,871125150794
6,Mansoura,870031111152
11,Sheikh Zayed,306511267245
8,Mohandseen,295659948124
4,Lauran,294751331072
2,Dokki,291078258307
12,Smouha,280805350268
7,Miami,279816470842
3,Heliopolis,228584497856


In [4]:
import plotly.express as px

# 1. Aggregate total revenue by city and sort descending
city_sales = (
    df.groupby('city')['final_price']
      .sum()
      .reset_index()
      .sort_values('final_price', ascending=False)
)

# 2. Format revenue into Billions (B) for clean chart labels
city_sales['label'] = (city_sales['final_price'] / 1e9).round(2).astype(str) + ' B'

# 3. Generate an interactive bar chart
fig = px.bar(
    city_sales,
    x='city',
    y='final_price',
    text='label',
    title='Total Revenue by City'
)

# 4. Enhance aesthetics (adjust label position and set brand color)
fig.update_traces(
    textposition='outside',
    marker_color="#9BC4F7"
)

fig.update_layout(
    title_x=0.5
)
fig.show()

In [5]:
# 1. Aggregate total revenue by city, sort descending, and extract the top 4 performers
city_sales = (
    df.groupby("city")["final_price"]
     .sum()
     .reset_index()
     .sort_values("final_price", ascending=False)
     .head(4)
)

# 2. Generate a horizontal bar chart with automated number formatting and a dynamic color gradient
fig_city = px.bar(
    city_sales,
    x="final_price",
    y="city",
    orientation="h",
    text_auto=".2s", # Automatically formats numbers with clean SI prefixes (e.g., M, B)
    color="final_price",
    color_continuous_scale="Blues",
    title="Top Cities by Revenue"
)

# 3. Center the chart title for a cleaner layout and display the plot
fig_city.update_layout(
    title_x=0.5
)

fig_city

In [6]:
# Aggregate total revenue (final_price) by year and month to analyze sales trends over time
monthly_sales = (
    df.groupby(['year', 'month'])['final_price']
      .sum()
      .reset_index()
)
monthly_sales

,year,month,final_price
0,2023,1,145870597848
1,2023,2,146677095483
2,2023,3,145469053126
3,2023,4,146123316327
4,2023,5,146601693331
5,2023,6,146356268894
6,2023,7,146157294042
7,2023,8,146260453248
8,2023,9,146199373128
9,2023,10,146720122983


In [7]:
# Aggregate total revenue (final_price) by year and month to analyze sales trends over time
monthly_sales = (
    df.groupby(['year', 'month'])['final_price']
      .sum()
      .reset_index()
)

# 1. Combine 'year' and 'month' into a single string column to create a continuous timeline for the X-axis
monthly_sales['year_month'] = (
    monthly_sales['year'].astype(str) + '-' +
    monthly_sales['month'].astype(str)
)

# 2. Generate a line chart with markers to visualize revenue trends and fluctuations over time
fig = px.line(
    monthly_sales,
    x='year_month',
    y='final_price',
    markers=True,
    title='Monthly Sales Trend'
)

fig.update_layout(
    title_x=0.5
)

fig.show()

In [8]:
import plotly.express as px

# 1. Aggregate revenue by project and extract the top 10 performers
project_sales = (
    df.groupby('project_name')['final_price']
      .sum()
      .reset_index()
      .sort_values('final_price', ascending=False)
      .head(10)
)

# 2. Generate a treemap to visualize proportional revenue contribution
fig_projects = px.treemap(
    project_sales,
    path=["project_name"],
    values="final_price",
    color="final_price",
    color_continuous_scale="Viridis",
    title="Revenue Contribution by Project",
    hover_data={
        "final_price": ":,.0f"
    }
)

# 3. Format text labels and hover tooltips to display bold names, exact revenue, and percentage share
fig_projects.update_traces(
    textinfo="label+percent parent",
    texttemplate=(
        "<b>%{label}</b><br>"
        "%{percentParent:.1%}"
    ),
    hovertemplate=(
        "<b>%{label}</b><br>"
        "Revenue: %{value:,.0f}<br>"
        "Share: %{percentParent:.1%}<extra></extra>"
    ),
    textfont_size=14
)

# 4. Clean up the layout (reduce margins to maximize chart space and center the title)
fig_projects.update_layout(
    margin=dict(t=60, l=0, r=0, b=0)
)


fig_projects.update_layout(
    title_x=0.5
)

fig_projects

In [9]:
# 1. Aggregate performance metrics per agent (calculating total revenue and total number of deals)
agent_perf = (
    df.groupby("agent")
    .agg(
        revenue=("final_price", "sum"),
        deals=("final_price", "count")
    )
    .reset_index()
)

# 2. Generate a bubble chart to evaluate agent efficiency (bubble size and color driven by revenue)
fig_agents = px.scatter(
    agent_perf,
    x="deals",
    y="revenue",
    size="revenue",
    hover_name="agent",
    color="revenue",
    size_max=40,
    title="Agent Performance: Deals vs Revenue"
)

# 3. Display the interactive chart
fig_agents

In [10]:
import plotly.express as px

# 1. Aggregate total revenue by customer segment
segment_sales = (
    df.groupby('segment')['final_price']
      .sum()
      .reset_index()
)

# 2. Convert revenue to Billions (B) for cleaner pie chart values
segment_sales['final_price_b'] = (segment_sales['final_price'] / 1e9).round(2)

# 3. Generate a pie chart to visualize revenue distribution across segments
fig = px.pie(
    segment_sales,
    names='segment',
    values='final_price_b',
    title='Revenue by Customer Segment',
    color='segment',
    color_discrete_sequence=px.colors.qualitative.Set2
)

# 4. Enhance aesthetics (show labels and percentages, format hover text, and center title)
fig.update_traces(
    textinfo='label+percent',
    hovertemplate='%{label}<br>%{value} B'
)
fig.update_layout(
    title_x=0.5
)
fig.show()

In [12]:
import plotly.express as px

# 1. Aggregate total revenue by sales channel and sort descending
channel_sales = (
    df.groupby('channel')['final_price']
      .sum()
      .reset_index()
      .sort_values('final_price', ascending=False)
)

# 2. Format revenue into Billions (B) for clean chart labels
channel_sales['label'] = (
    channel_sales['final_price'] / 1e9
).round(2).astype(str) + ' B'

# 3. Generate a vertical bar chart to compare performance across channels
fig = px.bar(
    channel_sales,
    x='channel',
    y='final_price',
    text='label',
    title='Revenue by Sales Channel'
)

# 4. Enhance aesthetics (adjust data label positioning and apply a distinct color)
fig.update_traces(
    textposition='outside',
    marker_color="#E4D07F" 
)
fig.update_layout(
    title_x=0.5
)

fig.show()

In [13]:
import plotly.express as px

# 1. Calculate the average deal size (mean revenue) per city and sort descending
avg_city_price = (
    df.groupby('city')['final_price']
      .mean()
      .reset_index()
      .sort_values('final_price', ascending=False)
)

# 2. Format the average values into Millions (M) for clean data labels
avg_city_price['label'] = (
    avg_city_price['final_price'] / 1e6
).round(2).astype(str) + ' M'

# 3. Generate a horizontal bar chart (ideal for readable city names)
fig = px.bar(
    avg_city_price,
    x='final_price',
    y='city',
    text='label',
    orientation='h',
    title='Average Deal Size per City'
)

# 4. Enhance aesthetics (adjust data label positioning and apply custom color)
fig.update_traces(
    textposition='outside',
    marker_color='#4F81BD'
)

fig.show()